# 📝 EXPERIMENT DOCUMENTATION & USAGE

print("""
# Context Window Size vs LLM Call Duration Experiment

## Overview
This experiment evaluates how LLM response time scales with context window size across
6 different models, testing context sizes from 10 tokens to 1 million tokens.

## Models Tested
1. **OpenAI GPT-5 Mini** (`openai/gpt-5-mini`)
2. **OpenAI GPT-4.1 Nano** (`openai/gpt-4.1-nano`)
3. **Google Gemini 2.5 Flash** (`google-ai/gemini-2.5-flash`)
4. **Google Gemini 2.0 Flash Lite** (`google-ai/gemini-2.0-flash-lite-001`)
5. **Claude Sonnet 4** (`azure/claude-sonnet-4-20250514`)
6. **Claude Haiku** (`anthropic/claude-3-haiku-20240307`)

## Token Sizes Tested
- 10 tokens (10¹)
- 100 tokens (10²)
- 1,000 tokens (10³)
- 10,000 tokens (10⁴)
- 100,000 tokens (10⁵)
- 1,000,000 tokens (10⁶)

## Methodology
- **Text Source**: Paul Graham essays + ArXiv papers (shuffled for variety)
- **Token Validation**: tiktoken encoding ensures exact token counts
- **Prompt**: Simple summarization task to focus on input processing
- **Iterations**: 3 per model/size combination for statistical reliability
- **Timing**: Wall-clock time including network latency
- **Error Handling**: Retry logic with exponential backoff

## Key Metrics
- **Duration**: Total response time (seconds)
- **Throughput**: Input tokens processed per second
- **Success Rate**: Percentage of successful API calls
- **Scaling Pattern**: How duration increases with context size

## Usage Instructions

### Quick Test (recommended first)
```python
# Test one model with small contexts
quick_results = run_context_window_experiment(
    models={"gpt-5-mini": MODELS["gpt-5-mini"]},
    token_sizes=[10, 100, 1000],
    iterations_per_test=2
)
```

### Full Experiment (108 tests, ~30-60 minutes)
```python
# Run the complete experiment (uncomment the input() line for confirmation)
full_results = run_context_window_experiment()
```

### Custom Experiment
```python
# Test specific models/sizes
custom_results = run_context_window_experiment(
    models={"claude-haiku": MODELS["claude-haiku"], 
            "gemini-2.5-flash": MODELS["gemini-2.5-flash"]},
    token_sizes=[1000, 10000, 100000],
    iterations_per_test=5
)
```

## Expected Results
- **Linear Scaling**: Most models show linear increase in processing time
- **Model Differences**: Speed varies significantly between providers
- **Context Limits**: Some models may fail at very large context sizes
- **Throughput Patterns**: Tokens/second may decrease with larger contexts

## Output Files
Results are automatically saved to `data/results/`:
- CSV: Structured data for analysis
- JSON: Complete results with metadata

## Troubleshooting
- **API Errors**: Check ORQ_API_KEY environment variable
- **Rate Limiting**: Experiment includes delays between calls
- **Memory Issues**: Large contexts may cause memory problems
- **Timeouts**: Very large contexts may exceed API timeouts

Run the cells above to start experimenting! 🚀
""")

# Quick reference for key functions
print("\n🔧 QUICK REFERENCE:")
print("- run_context_window_experiment(): Main experiment runner")
print("- analyze_experiment_results(): Convert to DataFrame with stats")
print("- visualize_results(): Create performance charts")
print("- save_experiment_results(): Export to CSV/JSON")
print("- create_text_with_exact_tokens(): Generate test text")
print("\n✅ Experiment setup complete! Ready to run.")

In [ ]:
# 📊 ANALYZE FULL EXPERIMENT RESULTS

if 'full_results' in locals() and full_results:
    print("🎯 Analyzing full experiment results...")
    
    # Comprehensive analysis
    df_full = analyze_experiment_results(full_results)
    
    # Create visualizations
    visualize_results(df_full)
    
    # Save results
    csv_path, json_path = save_experiment_results(full_results, "full_experiment")
    
    print(f"\n🏆 EXPERIMENT COMPLETE!")
    print(f"📁 Results saved:")
    print(f"  - CSV: {csv_path}")
    print(f"  - JSON: {json_path}")
    
    # Summary insights
    successful_df = df_full[df_full['success']].copy()
    if len(successful_df) > 0:
        print(f"\n💡 KEY INSIGHTS:")
        print(f"- Fastest model: {successful_df.loc[successful_df['duration_seconds'].idxmin(), 'model_name']}")
        print(f"- Highest throughput: {successful_df.loc[successful_df['tokens_per_second'].idxmax(), 'model_name']}")
        print(f"- Most reliable: {df_full.groupby('model_name')['success'].mean().idxmax()}")
        
        # Context size analysis
        size_performance = successful_df.groupby('context_size')['duration_seconds'].mean()
        print(f"- Largest context handled: {successful_df['context_size'].max():,} tokens")
        print(f"- Average scaling factor (10K vs 1K tokens): {size_performance[10000] / size_performance[1000]:.1f}x slower")
    
else:
    print("❌ No full experiment results available. Run the full experiment first!")

In [ ]:
# 🚀 FULL EXPERIMENT: Run all models across all context sizes
# 
# This cell runs the complete experiment with all 6 models across powers of 10 token sizes.
# Warning: This will take significant time (potentially hours) and make many API calls.
# 
# Models tested:
# - openai/gpt-5-mini
# - openai/gpt-4.1-nano  
# - google-ai/gemini-2.5-flash
# - google-ai/gemini-2.0-flash-lite-001
# - azure/claude-sonnet-4-20250514
# - anthropic/claude-3-haiku-20240307
#
# Token sizes: 10, 100, 1K, 10K, 100K, 1M tokens
# Iterations per combination: 3 (for statistical reliability)
# Total tests: 6 models × 6 sizes × 3 iterations = 108 tests

print("🔬 FULL CONTEXT WINDOW SCALING EXPERIMENT")
print("=" * 60)
print("This experiment will:")
print("- Test 6 different LLM models")
print("- Across 6 context sizes (10¹ to 10⁶ tokens)")  
print("- With 3 iterations per combination")
print("- Total: ~108 API calls")
print("- Estimated time: 30-60 minutes")
print("\nPress Enter to start, or Ctrl+C to cancel...")
# input()  # Uncomment to require confirmation

# Run the full experiment
full_results = run_context_window_experiment(
    models=MODELS,  # All 6 models
    token_sizes=[10, 100, 1000, 10000, 100000, 1000000],  # Powers of 10
    iterations_per_test=3,  # 3 iterations for reliability
    sentences=sentences
)

In [ ]:
def save_experiment_results(results: List[ExperimentResult], filename: str = None):
    """Save experiment results to JSON and CSV files."""
    
    if filename is None:
        timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        filename = f"context_window_experiment_{timestamp}"
    
    results_dir = DATA_DIR / "results"
    results_dir.mkdir(exist_ok=True)
    
    # Convert to DataFrame
    df = analyze_experiment_results(results)
    
    # Save as CSV
    csv_path = results_dir / f"{filename}.csv"
    df.to_csv(csv_path, index=False)
    print(f"💾 Results saved to CSV: {csv_path}")
    
    # Save as JSON with full details
    json_data = []
    for result in results:
        json_data.append({
            "model_name": result.model_name,
            "context_size": result.context_size,
            "duration_seconds": result.duration_seconds,
            "tokens_per_second": result.tokens_per_second,
            "success": result.success,
            "error_message": result.error_message,
            "timestamp": pd.Timestamp.now().isoformat()
        })
    
    json_path = results_dir / f"{filename}.json"
    with open(json_path, 'w') as f:
        json.dump(json_data, f, indent=2)
    print(f"💾 Results saved to JSON: {json_path}")
    
    return csv_path, json_path

# Save initial test results if available
if 'results' in locals() and results:
    save_experiment_results(results, "initial_test")

print("✅ Data analysis and visualization code ready!")

In [ ]:
def analyze_experiment_results(results: List[ExperimentResult]) -> pd.DataFrame:
    """Convert experiment results to pandas DataFrame and perform analysis."""
    
    # Convert to DataFrame
    data = []
    for result in results:
        data.append({
            'model_name': result.model_name,
            'context_size': result.context_size,
            'duration_seconds': result.duration_seconds if result.success else None,
            'tokens_per_second': result.tokens_per_second if result.success else None,
            'success': result.success,
            'error_message': result.error_message
        })
    
    df = pd.DataFrame(data)
    
    # Print summary statistics
    print("📊 EXPERIMENT RESULTS SUMMARY")
    print("=" * 50)
    
    print(f"\nTotal tests: {len(df)}")
    print(f"Successful tests: {df['success'].sum()}")
    print(f"Failed tests: {(~df['success']).sum()}")
    print(f"Success rate: {df['success'].mean():.1%}")
    
    # Success rate by model
    print("\n🏆 Success Rate by Model:")
    success_by_model = df.groupby('model_name')['success'].agg(['count', 'sum', 'mean'])
    success_by_model.columns = ['total_tests', 'successful_tests', 'success_rate']
    success_by_model['success_rate'] = success_by_model['success_rate'].apply(lambda x: f"{x:.1%}")
    print(success_by_model)
    
    # Performance analysis for successful tests only
    successful_df = df[df['success']].copy()
    
    if len(successful_df) > 0:
        print("\n⚡ Performance Analysis (Successful Tests Only):")
        print(f"Average duration: {successful_df['duration_seconds'].mean():.2f}s")
        print(f"Average throughput: {successful_df['tokens_per_second'].mean():.1f} tokens/sec")
        
        print("\n📈 Performance by Context Size:")
        perf_by_size = successful_df.groupby('context_size').agg({
            'duration_seconds': ['mean', 'std', 'count'],
            'tokens_per_second': ['mean', 'std']
        }).round(2)
        print(perf_by_size)
        
        print("\n🚀 Performance by Model:")
        perf_by_model = successful_df.groupby('model_name').agg({
            'duration_seconds': ['mean', 'std', 'count'],
            'tokens_per_second': ['mean', 'std']
        }).round(2)
        print(perf_by_model)
    
    # Error analysis
    failed_df = df[~df['success']].copy()
    if len(failed_df) > 0:
        print(f"\n❌ Error Analysis ({len(failed_df)} failures):")
        error_counts = failed_df['error_message'].value_counts()
        for error, count in error_counts.head(5).items():
            print(f"  - {error}: {count} occurrences")
    
    return df

def visualize_results(df: pd.DataFrame):
    """Create visualizations of the experiment results."""
    
    successful_df = df[df['success']].copy()
    
    if len(successful_df) == 0:
        print("❌ No successful results to visualize")
        return
    
    # Set up the plotting style
    plt.style.use('default')
    sns.set_palette("husl")
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Context Window Size vs LLM Performance Analysis', fontsize=16, fontweight='bold')
    
    # 1. Duration vs Context Size (log scale)
    ax1 = axes[0, 0]
    for model in successful_df['model_name'].unique():
        model_data = successful_df[successful_df['model_name'] == model]
        ax1.scatter(model_data['context_size'], model_data['duration_seconds'], 
                   label=model, alpha=0.7, s=60)
    
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_xlabel('Context Size (tokens)')
    ax1.set_ylabel('Duration (seconds)')
    ax1.set_title('Response Time vs Context Size')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Tokens per Second vs Context Size
    ax2 = axes[0, 1]
    for model in successful_df['model_name'].unique():
        model_data = successful_df[successful_df['model_name'] == model]
        ax2.scatter(model_data['context_size'], model_data['tokens_per_second'], 
                   label=model, alpha=0.7, s=60)
    
    ax2.set_xscale('log')
    ax2.set_xlabel('Context Size (tokens)')
    ax2.set_ylabel('Throughput (tokens/sec)')
    ax2.set_title('Processing Throughput vs Context Size')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Box plot of Duration by Model
    ax3 = axes[1, 0]
    if len(successful_df['model_name'].unique()) > 1:
        sns.boxplot(data=successful_df, x='model_name', y='duration_seconds', ax=ax3)
        ax3.set_title('Duration Distribution by Model')
        ax3.set_xlabel('Model')
        ax3.set_ylabel('Duration (seconds)')
        ax3.tick_params(axis='x', rotation=45)
    else:
        ax3.text(0.5, 0.5, 'Multiple models needed\nfor comparison', 
                ha='center', va='center', transform=ax3.transAxes)
        ax3.set_title('Duration Distribution by Model')
    
    # 4. Efficiency scatter: Duration vs Tokens per Second
    ax4 = axes[1, 1]
    for model in successful_df['model_name'].unique():
        model_data = successful_df[successful_df['model_name'] == model]
        ax4.scatter(model_data['duration_seconds'], model_data['tokens_per_second'], 
                   label=model, alpha=0.7, s=60)
    
    ax4.set_xlabel('Duration (seconds)')
    ax4.set_ylabel('Throughput (tokens/sec)')
    ax4.set_title('Speed vs Throughput Efficiency')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Additional scaling analysis
    if len(successful_df) > 3:
        print("\n📊 Scaling Pattern Analysis:")
        print("-" * 40)
        
        for model in successful_df['model_name'].unique():
            model_data = successful_df[successful_df['model_name'] == model].sort_values('context_size')
            if len(model_data) >= 3:
                # Simple correlation analysis
                import numpy as np
                log_size = np.log10(model_data['context_size'])
                log_duration = np.log10(model_data['duration_seconds'])
                correlation = np.corrcoef(log_size, log_duration)[0, 1]
                
                print(f"{model}:")
                print(f"  - Log-log correlation: {correlation:.3f}")
                if correlation > 0.8:
                    print(f"  - Pattern: Strong positive scaling (slower with more context)")
                elif correlation > 0.5:
                    print(f"  - Pattern: Moderate positive scaling")
                else:
                    print(f"  - Pattern: Weak or complex scaling relationship")

# Analyze the initial test results
if 'results' in locals() and results:
    df_results = analyze_experiment_results(results)
    visualize_results(df_results)
else:
    print("No results available yet. Run the experiment first!")

In [ ]:
def run_context_window_experiment(
    models: Dict[str, str] = MODELS,
    token_sizes: List[int] = [10, 100, 1000, 10000, 100000, 1000000],
    iterations_per_test: int = 3,
    sentences: List[str] = None
) -> List[ExperimentResult]:
    """Run the full context window scaling experiment."""
    
    if sentences is None:
        sentences = prepare_text_data_for_experiments()
    
    all_results = []
    total_tests = len(models) * len(token_sizes) * iterations_per_test
    
    print(f"Starting context window experiment:")
    print(f"- Models: {len(models)} ({list(models.keys())})")
    print(f"- Token sizes: {token_sizes}")
    print(f"- Iterations per test: {iterations_per_test}")
    print(f"- Total tests: {total_tests}")
    print("=" * 60)
    
    test_count = 0
    
    for model_name, model_id in models.items():
        print(f"\n🧪 Testing {model_name} ({model_id})")
        print("-" * 50)
        
        for token_count in token_sizes:
            print(f"\n📊 Token size: {token_count:,}")
            
            # Run multiple iterations for this model/token size combo
            iteration_results = []
            for iteration in range(iterations_per_test):
                test_count += 1
                print(f"  Iteration {iteration + 1}/{iterations_per_test} (Test {test_count}/{total_tests})")
                
                try:
                    result = run_single_experiment(model_name, model_id, token_count, sentences)
                    iteration_results.append(result)
                    all_results.append(result)
                    
                    if result.success:
                        print(f"    ✓ {result.duration_seconds:.2f}s ({result.tokens_per_second:.1f} tokens/sec)")
                    else:
                        print(f"    ✗ Failed: {result.error_message}")
                        
                except Exception as e:
                    print(f"    ⚠️  Unexpected error: {e}")
                    error_result = ExperimentResult(
                        model_name=model_name,
                        context_size=token_count,
                        duration_seconds=0.0,
                        tokens_per_second=0.0,
                        success=False,
                        error_message=str(e)
                    )
                    iteration_results.append(error_result)
                    all_results.append(error_result)
                
                # Small delay between tests to avoid rate limiting
                time.sleep(1)
            
            # Show summary for this token size
            successful_results = [r for r in iteration_results if r.success]
            if successful_results:
                avg_duration = sum(r.duration_seconds for r in successful_results) / len(successful_results)
                avg_tokens_per_sec = sum(r.tokens_per_second for r in successful_results) / len(successful_results)
                print(f"  📈 Average: {avg_duration:.2f}s ({avg_tokens_per_sec:.1f} tokens/sec)")
            else:
                print(f"  ❌ All iterations failed for {token_count:,} tokens")
    
    print("\n" + "=" * 60)
    print(f"🏁 Experiment complete! {len(all_results)} total tests run.")
    
    # Quick summary
    successful_tests = [r for r in all_results if r.success]
    print(f"✅ Successful: {len(successful_tests)}/{len(all_results)} ({len(successful_tests)/len(all_results)*100:.1f}%)")
    
    return all_results

# Run the experiment (start with a subset for testing)
print("🚀 Starting Context Window Scaling Experiment...")
results = run_context_window_experiment(
    models={"gpt-5-mini": MODELS["gpt-5-mini"]},  # Start with just one model for testing
    token_sizes=[10, 100, 1000],  # Start with smaller sizes
    iterations_per_test=2,  # Fewer iterations for initial test
    sentences=sentences
)

In [ ]:
def time_llm_call(model_name: str, model_id: str, context_text: str, max_retries: int = 3) -> ExperimentResult:
    """Time a single LLM call and return results."""
    
    # Simple prompt that incorporates the context
    prompt = f"""Based on the following context, provide a brief summary in 1-2 sentences:

Context:
{context_text}

Summary:"""
    
    context_tokens = len(encoding.encode(context_text))
    
    for attempt in range(max_retries):
        try:
            start_time = time.time()
            
            response = orq_client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=50,  # Short response to focus on input processing time
                temperature=0.0
            )
            
            end_time = time.time()
            duration = end_time - start_time
            tokens_per_sec = context_tokens / duration if duration > 0 else 0
            
            return ExperimentResult(
                model_name=model_name,
                context_size=context_tokens,
                duration_seconds=duration,
                tokens_per_second=tokens_per_sec,
                success=True
            )
            
        except Exception as e:
            error_msg = str(e)
            print(f"Attempt {attempt + 1} failed for {model_name} with {context_tokens} tokens: {error_msg}")
            
            if attempt == max_retries - 1:
                return ExperimentResult(
                    model_name=model_name,
                    context_size=context_tokens,
                    duration_seconds=0.0,
                    tokens_per_second=0.0,
                    success=False,
                    error_message=error_msg
                )
            
            # Wait before retry
            time.sleep(2 ** attempt)

def run_single_experiment(model_name: str, model_id: str, token_count: int, sentences: List[str]) -> ExperimentResult:
    """Run a single experiment with specified model and token count."""
    print(f"Testing {model_name} with {token_count:,} tokens...")
    
    # Create text with exact token count
    context_text = create_text_with_exact_tokens(token_count, sentences)
    
    # Time the LLM call
    result = time_llm_call(model_name, model_id, context_text)
    
    if result.success:
        print(f"✓ {model_name}: {result.duration_seconds:.2f}s ({result.tokens_per_second:.1f} tokens/sec)")
    else:
        print(f"✗ {model_name}: Failed - {result.error_message}")
    
    return result

# Test the timing framework with a small example
print("Testing timing framework...")
test_result = run_single_experiment("gpt-5-mini", MODELS["gpt-5-mini"], 100, sentences)

In [ ]:
# Configure LLM clients for all models via ORQ proxy
ORQ_API_KEY = os.getenv("ORQ_API_KEY")
assert ORQ_API_KEY is not None, "ORQ_API_KEY is not set"

ORQ_BASE_URL = "https://api.orq.ai/v2/proxy"

# Model configurations with exact ORQ proxy names
MODELS = {
    "gpt-5-mini": "openai/gpt-5-mini",
    "gpt-4.1-nano": "openai/gpt-4.1-nano", 
    "gemini-2.5-flash": "google-ai/gemini-2.5-flash",
    "gemini-2.0-flash-lite": "google-ai/gemini-2.0-flash-lite-001",
    "claude-sonnet-4": "azure/claude-sonnet-4-20250514",
    "claude-haiku": "anthropic/claude-3-haiku-20240307"
}

# Create OpenAI client for ORQ proxy (handles OpenAI, Google, Anthropic models)
orq_client = OpenAI(
    api_key=ORQ_API_KEY,
    base_url=ORQ_BASE_URL
)

print("Configured ORQ client for all 6 models:")
for name, model_id in MODELS.items():
    print(f"  - {name}: {model_id}")

In [ ]:
def prepare_text_data_for_experiments():
    """Load and prepare text data from various sources for context window experiments."""
    
    # Load Paul Graham essays
    pg_essays = []
    pg_path = DATA_DIR / "paul_graham_essays"
    if pg_path.exists():
        for txt_file in pg_path.glob("*.txt"):
            with open(txt_file, 'r', encoding='utf-8') as f:
                pg_essays.append(f.read())
    
    # Load arxiv papers
    arxiv_papers = []
    arxiv_path = DATA_DIR / "arxiv_papers"
    if arxiv_path.exists():
        for txt_file in list(arxiv_path.glob("*.txt"))[:20]:  # Limit to 20 papers for performance
            with open(txt_file, 'r', encoding='utf-8') as f:
                arxiv_papers.append(f.read())
    
    # Combine all text sources
    all_text = " ".join(pg_essays + arxiv_papers)
    
    # Split into sentences for better mixing
    sentences = all_text.replace('\n', ' ').split('. ')
    sentences = [s.strip() + '.' for s in sentences if len(s.strip()) > 10]
    
    print(f"Prepared {len(sentences)} sentences from {len(pg_essays)} PG essays and {len(arxiv_papers)} arxiv papers")
    return sentences

def create_text_with_exact_tokens(target_tokens: int, sentences: List[str]) -> str:
    """Create text with exactly the specified number of tokens using tiktoken validation."""
    
    current_text = ""
    current_tokens = 0
    
    # Randomly shuffle sentences to create variety
    shuffled_sentences = sentences.copy()
    random.shuffle(shuffled_sentences)
    
    # Add sentences until we're close to target
    sentence_idx = 0
    while current_tokens < target_tokens - 100:  # Leave some buffer
        if sentence_idx >= len(shuffled_sentences):
            sentence_idx = 0
            random.shuffle(shuffled_sentences)
        
        sentence = shuffled_sentences[sentence_idx]
        sentence_tokens = len(encoding.encode(sentence))
        
        if current_tokens + sentence_tokens <= target_tokens:
            current_text += " " + sentence
            current_tokens += sentence_tokens
        
        sentence_idx += 1
    
    # Fine-tune to exact token count
    while current_tokens < target_tokens:
        current_text += " word"
        current_tokens = len(encoding.encode(current_text))
    
    while current_tokens > target_tokens:
        current_text = current_text[:-1]
        current_tokens = len(encoding.encode(current_text))
    
    # Final validation
    final_tokens = len(encoding.encode(current_text))
    print(f"Created text with {final_tokens} tokens (target: {target_tokens})")
    
    return current_text.strip()

# Test the text preparation
sentences = prepare_text_data_for_experiments()

In [ ]:
# Context Window Size vs LLM Duration Experiment
import time
import random
from typing import List, Dict, Any, Tuple
from openai import OpenAI
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

@dataclass
class ExperimentResult:
    model_name: str
    context_size: int
    duration_seconds: float
    tokens_per_second: float
    success: bool
    error_message: str = None

In [ ]:
import polars as pl
from pathlib import Path
import tiktoken

DATA_DIR = Path("/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data")
(DATA_DIR / "processed").mkdir(parents=True, exist_ok=True)

encoding = tiktoken.encoding_for_model("gpt-4o")

df = pl.scan_parquet(DATA_DIR / "processed/nq_question_answer.parquet")
df = df.with_columns(
    pl.col("document_html").str.len_chars().alias("context_length"),
    pl.col("document_html").map_elements(lambda x: len(encoding.encode(x)), return_dtype=pl.Int64).alias("token_count"),
).collect()

In [2]:
df.select(
    pl.col("token_count").mean().alias("avg_tokens"),
    pl.col("token_count").max().alias("max_tokens"),
    pl.col("token_count").std().alias("min_tokens"),
)

avg_tokens,max_tokens,min_tokens
f64,i64,f64
76573.488378,548918,54611.959385
